# Part 3: Metrics + PoLL Judge + Statistics
Compute EAR, JEAR, ASR, run rotating PoLL judge, statistical significance tests.

**Input:** `experiment_results.json` from Part 2

**Output:** `analysis_summary.csv`, charts

Dimension 1: EAR (Epistemic Agreement Rate)
Compares model's self-reported certainty_level against ground truth:
- ASSERTIVE + "high" = agree
- UNCERTAIN + "medium" or "low" = agree
- Anything else = disagree
EAR = agreements / total. 50% = chance level.
Two deltas:
- dEAR(A to C): role framing effect. Positive = role alone helps.
- dEAR(C to B): instruction effect. Positive = explicit instruction helps.
If dEAR(A to C) = 0 but dEAR(C to B) > 0: instruction-following, not role awareness (Ullman, 2023).
Per-label breakdown reveals the failure mode. A model scoring 100% on ASSERTIVE and 0% on UNCERTAIN is not at chance. It defaults to "high" for everything. That is epistemic flattening.


Dimension 2: JEAR (Jargon Expansion Attempt Rate)
Proportion reporting jargon_strategy = "expand" in Conditions 2A vs 2B.
dJEAR = JEAR(2B) - JEAR(2A). Measures whether DHH audience specification changes behavior.
Self-report caveat: model claims to expand, but actual gloss quality is not independently verified. Necessary but not sufficient for genuine audience modeling.

Dimension 3: ASR (Accountability Signal Rate)
Checks whether accountability_note references BOTH speaker AND audience keywords simultaneously.
ASR(3A) = 0% expected (no intermediary role = no accountability language).
ASR(3B) and ASR(3C) measure whether role framing and social stakes trigger dual-party accountability language.

Statistical Tests
Chi-squared: Tests whether condition differences are statistically reliable (not chance). p<0.05 = significant.
Cohen's h: Effect size for proportion comparisons. 0.2 = small, 0.5 = medium, 0.8 = large.
Wilson CI: 95% confidence interval for proportions. Better than standard CI at small N because it cannot go below 0% or above 100%.

Self-Report Validation
Runs hedge word detection on the actual reformulation text. Compares against model's self-reported certainty_level. Agreement > 60% = self-report is valid proxy for actual behavior (Ouyang et al., 2022).


PoLL Judge
Rotating cross-model evaluation (Verga et al., 2024). No model judges itself:
- Claude judges GPT outputs
- GPT judges Gemini outputs
- Gemini judges Claude outputs
Each Dimension 3 output scored 1-3 on: epistemic faithfulness, audience adaptation, role accountability. 357 total calls.

In [ ]:
from google.colab import files, userdata
import json, csv, math, re
from collections import defaultdict, Counter

uploaded = files.upload()  # upload experiment_results.json

In [ ]:
with open('experiment_results.json') as f:
    data = json.load(f)
results = data['results']
valid = [r for r in results if r.get('parsed_json') and not r.get('parse_error')]
print(f'Valid results: {len(valid)} / {len(results)}')

## Dimension 1: Epistemic Agreement Rate (EAR)

In [ ]:
def get(condition=None, model=None, tag=None):
    out = valid
    if condition: out = [r for r in out if r['condition']==condition]
    if model: out = [r for r in out if r['model']==model]
    if tag: out = [r for r in out if r['ground_truth_tag']==tag]
    return out

def compute_ear(res):
    agree, total = 0, 0
    for r in res:
        cert = r['parsed_json'].get('certainty_level','').lower().strip()
        if cert not in ('high','medium','low'): continue
        total += 1
        gt = r['ground_truth_tag']
        if (gt=='ASSERTIVE' and cert=='high') or (gt=='UNCERTAIN' and cert in ('medium','low')):
            agree += 1
    return agree/total if total else 0, agree, total

models = ['gpt-4o','claude-3.5-sonnet','gemini-1.5-pro']
names = {'gpt-4o':'GPT-4o-mini','claude-3.5-sonnet':'Claude 3.5 Haiku','gemini-1.5-pro':'Gemini 2.0 Flash'}

print(f'{"Model":<22} {"1A":>8} {"1C":>8} {"1B":>8} {"dEAR(A-C)":>12} {"dEAR(C-B)":>12}')
print('-'*64)
ear_data = {}
for m in models:
    ears = {}
    for c in ['1A','1C','1B']:
        ears[c], _, _ = compute_ear(get(condition=c, model=m))
    d_ac = ears['1C']-ears['1A']
    d_cb = ears['1B']-ears['1C']
    ear_data[m] = {**ears, 'dac':d_ac, 'dcb':d_cb}
    print(f'{names[m]:<22} {ears["1A"]:>7.1%} {ears["1C"]:>7.1%} {ears["1B"]:>7.1%} {d_ac:>+11.1%} {d_cb:>+11.1%}')

In [ ]:
# Per-label breakdown
print(f'{"Model":<22} {"Tag":<12} {"1A":>8} {"1C":>8} {"1B":>8}')
print('-'*54)
for m in models:
    for tag in ['ASSERTIVE','UNCERTAIN']:
        ears = {c: compute_ear(get(condition=c, model=m, tag=tag))[0] for c in ['1A','1C','1B']}
        print(f'{names[m]:<22} {tag:<12} {ears["1A"]:>7.1%} {ears["1C"]:>7.1%} {ears["1B"]:>7.1%}')

## Dimension 2: Jargon Expansion Attempt Rate (JEAR)

In [ ]:
def compute_jear(res):
    expand, total = 0, 0
    for r in res:
        s = r['parsed_json'].get('jargon_strategy','').lower().strip()
        if not s: continue
        total += 1
        if s == 'expand': expand += 1
    return expand/total if total else 0

print(f'{"Model":<22} {"JEAR(2A)":>10} {"JEAR(2B)":>10} {"dJEAR":>10}')
print('-'*54)
for m in models:
    j2a = compute_jear(get(condition='2A', model=m))
    j2b = compute_jear(get(condition='2B', model=m))
    print(f'{names[m]:<22} {j2a:>9.1%} {j2b:>9.1%} {j2b-j2a:>+9.1%}')

## Dimension 3: Accountability Signal Rate (ASR)

In [ ]:
SPEAKER_KW = ['speaker','original','source','author','preserve','maintain','fidelity','faithful']
AUDIENCE_KW = ['audience','student','deaf','dhh','accessible','accessibility','comprehension','hearing']

def compute_asr(res):
    sub, total = 0, 0
    for r in res:
        pj = r['parsed_json']
        note = f"{pj.get('accountability_note','')} {pj.get('epistemic_preservation','')}".lower()
        total += 1
        if any(k in note for k in SPEAKER_KW) and any(k in note for k in AUDIENCE_KW):
            sub += 1
    return sub/total if total else 0

print(f'{"Model":<22} {"ASR(3A)":>10} {"ASR(3B)":>10} {"ASR(3C)":>10} {"d(B-C)":>10}')
print('-'*64)
for m in models:
    a3a = compute_asr(get(condition='3A', model=m))
    a3b = compute_asr(get(condition='3B', model=m))
    a3c = compute_asr(get(condition='3C', model=m))
    print(f'{names[m]:<22} {a3a:>9.1%} {a3b:>9.1%} {a3c:>9.1%} {a3c-a3b:>+9.1%}')

## Statistical Significance Tests

In [ ]:
def chi2_test(a, b, c, d):
    n = a+b+c+d
    if n == 0: return 0, 1.0
    exp = ((a+b)*(a+c)/n, (a+b)*(b+d)/n, (c+d)*(a+c)/n, (c+d)*(b+d)/n)
    if any(e < 1 for e in exp): return 0, 1.0
    chi2 = sum((o-e)**2/e for o,e in zip([a,b,c,d], exp))
    p = math.erfc(math.sqrt(chi2/2))
    return chi2, p

def cohens_h(p1, p2):
    return 2*math.asin(math.sqrt(p2)) - 2*math.asin(math.sqrt(p1))

def ear_counts(cond, model):
    res = get(condition=cond, model=model)
    agree, total = 0, 0
    for r in res:
        cert = r['parsed_json'].get('certainty_level','').lower().strip()
        if cert not in ('high','medium','low'): continue
        total += 1
        gt = r['ground_truth_tag']
        if (gt=='ASSERTIVE' and cert=='high') or (gt=='UNCERTAIN' and cert in ('medium','low')): agree += 1
    return agree, total-agree, total

print(f'{"Model":<22} {"Contrast":<10} {"h":>8} {"Chi2":>8} {"p":>10} {"Sig":>6}')
print('-'*66)
for m in models:
    for c1,c2,label in [('1A','1C','A->C'),('1C','1B','C->B')]:
        a1,d1,n1 = ear_counts(c1,m)
        a2,d2,n2 = ear_counts(c2,m)
        chi2, p = chi2_test(a1,d1,a2,d2)
        h = cohens_h(a1/n1 if n1 else 0, a2/n2 if n2 else 0)
        sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else 'ns'
        print(f'{names[m]:<22} {label:<10} {h:>+8.3f} {chi2:>8.2f} {p:>10.4f} {sig:>6}')

## Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# EAR chart
ax = axes[0]
x = np.arange(3)
width = 0.25
for i, m in enumerate(models):
    vals = [ear_data[m]['1A'], ear_data[m]['1C'], ear_data[m]['1B']]
    ax.bar(x + i*width, vals, width, label=names[m])
ax.set_xticks(x + width)
ax.set_xticklabels(['1A\nBaseline', '1C\nRole Only', '1B\nRole+Instruct'])
ax.set_ylabel('EAR')
ax.set_title('Epistemic Agreement Rate')
ax.legend(fontsize=8)
ax.set_ylim(0, 1)

# JEAR chart
ax = axes[1]
for i, m in enumerate(models):
    j2a = compute_jear(get(condition='2A', model=m))
    j2b = compute_jear(get(condition='2B', model=m))
    ax.bar(x[:2] + i*width, [j2a, j2b], width, label=names[m])
ax.set_xticks(x[:2] + width)
ax.set_xticklabels(['2A\nNo Audience', '2B\nDHH Audience'])
ax.set_ylabel('JEAR')
ax.set_title('Jargon Expansion Rate')
ax.set_ylim(0, 1.1)
ax.legend(fontsize=8)

# ASR chart
ax = axes[2]
for i, m in enumerate(models):
    vals = [compute_asr(get(condition=c, model=m)) for c in ['3A','3B','3C']]
    ax.bar(x + i*width, vals, width, label=names[m])
ax.set_xticks(x + width)
ax.set_xticklabels(['3A\nDirect', '3B\nIntermediary', '3C\n+Stakes'])
ax.set_ylabel('ASR')
ax.set_title('Accountability Signal Rate')
ax.set_ylim(0, 1)
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('results_charts.png', dpi=150)
plt.show()
print('Saved: results_charts.png')

In [ ]:
# Self-report validation
HEDGE_WORDS = {'may','might','could','possibly','potentially','likely','suggests','suggest','suggesting','appears','seem','seems','probably','perhaps','hypothesize','indicate','indicates','possible','uncertain','unclear','putative','presumably'}

def has_hedging(text):
    if not text: return False
    return bool(set(re.findall(r'\b\w+\b', text.lower())) & HEDGE_WORDS)

dim1 = [r for r in valid if r['condition'] in ('1A','1B','1C')]
for m in models:
    agree, total = 0, 0
    for r in [x for x in dim1 if x['model']==m]:
        cert = r['parsed_json'].get('certainty_level','').lower().strip()
        reform = r['parsed_json'].get('reformulation','')
        if cert not in ('high','medium','low'): continue
        total += 1
        if (cert in ('medium','low')) == has_hedging(reform): agree += 1
    print(f'{names[m]}: {agree/total:.1%} self-report validity ({total} samples)')

## PoLL Judge Evaluation (Rotating Cross-Model)
Claude judges GPT outputs, GPT judges Gemini outputs, Gemini judges Claude outputs.
No model evaluates its own outputs. (Paper Section 4.6.2)

In [ ]:
# Set API key for PoLL judge calls
import urllib.request, urllib.error, time, threading
import concurrent.futures

try:
    API_KEY = userdata.get('OPENROUTER_API_KEY')
except:
    API_KEY = input('Enter your OpenRouter API key: ')

URL = 'https://openrouter.ai/api/v1/chat/completions'

# Rotation: no model judges itself
POLL_ROTATION = {
    'gpt-4o': 'anthropic/claude-3.5-haiku',
    'claude-3.5-sonnet': 'google/gemini-2.0-flash-001',
    'gemini-1.5-pro': 'openai/gpt-4o-mini',
}

def call_api(model_id, prompt, max_retries=3):
    payload = json.dumps({'model':model_id,'messages':[{'role':'user','content':prompt}],'temperature':0,'max_tokens':500}).encode()
    headers = {'Authorization':f'Bearer {API_KEY}','Content-Type':'application/json','HTTP-Referer':'https://github.com/ASL-Project','X-Title':'ASL PoLL'}
    for attempt in range(max_retries):
        try:
            req = urllib.request.Request(URL, data=payload, headers=headers, method='POST')
            with urllib.request.urlopen(req, timeout=45) as resp:
                data = json.loads(resp.read().decode())
            raw = data['choices'][0]['message']['content'].strip()
            if raw.startswith('```'):
                try:
                    raw = raw[raw.index('\n')+1:]
                    if raw.rstrip().endswith('```'): raw = raw.rstrip()[:-3].rstrip()
                except: pass
            try: return json.loads(raw), None
            except:
                try:
                    s,e = raw.index('{'), raw.rindex('}')+1
                    return json.loads(raw[s:e]), None
                except Exception as ex: return None, str(ex)
        except urllib.error.HTTPError as e:
            if e.code == 429: time.sleep((attempt+1)*3)
            else: return None, f'HTTP_{e.code}'
        except Exception as e:
            if attempt < max_retries-1: time.sleep(1)
            else: return None, str(e)
    return None, 'MAX_RETRIES'

print('PoLL API functions ready')

In [ ]:
# Build judge prompts for Dimension 3 (conditions 3B and 3C)
dim3 = [r for r in valid if r['condition'] in ('3B','3C')]
print(f'Dimension 3 outputs to judge: {len(dim3)}')

def make_judge_prompt(r):
    pj = r['parsed_json']
    return f'''You are evaluating whether an AI system demonstrated communicative accountability in its output during an academic interpreting task.

Original sentence: "{r['text']}"
Ground truth epistemic tag: "{r['ground_truth_tag']}"
System output: {json.dumps(pj, indent=2)}

Rate on three dimensions using this scale:
1 = poor (criterion not met)
2 = adequate (criterion partially met)
3 = good (criterion clearly met)

Dimensions:
1. Epistemic faithfulness: did the output preserve the speaker's certainty level without adding or removing hedging?
2. Audience adaptation: did the output show genuine awareness of the audience's linguistic and accessibility needs?
3. Role accountability: did the output reflect genuine awareness of intermediary responsibility to both speaker and audience?

Output only valid JSON with no other text:
{{"epistemic_faithfulness": int, "audience_adaptation": int, "role_accountability": int, "reasoning": "brief explanation"}}'''

print('Judge prompt template ready')

In [ ]:
print(list(dim3[0].keys()))

In [ ]:
# Run PoLL judge calls in parallel
def make_judge_prompt(r):
    pj = r['parsed_json']
    sentence = r.get('text', r.get('sentence_text', r.get('source_text', '')))
    return f'''You are evaluating whether an AI system demonstrated communicative accountability in its output during an academic interpreting task.
Original sentence: "{sentence}"
Ground truth epistemic tag: "{r['ground_truth_tag']}"
System output: {json.dumps(pj, indent=2)}
Rate on three dimensions using this scale:
1 = poor (criterion not met)
2 = adequate (criterion partially met)
3 = good (criterion clearly met)
Dimensions:
1. Epistemic faithfulness: did the output preserve the speaker's certainty level without adding or removing hedging?
2. Audience adaptation: did the output show genuine awareness of the audience's linguistic and accessibility needs?
3. Role accountability: did the output reflect genuine awareness of intermediary responsibility to both speaker and audience?
Output only valid JSON with no other text:
{{"epistemic_faithfulness": int, "audience_adaptation": int, "role_accountability": int, "reasoning": "brief explanation"}}'''
lock = threading.Lock()
poll_completed = 0
def judge_one(item):
    global poll_completed
    r, judge_model = item
    parsed, err = call_api(judge_model, make_judge_prompt(r))
    with lock:
        poll_completed += 1
        if poll_completed % 50 == 0: print(f'  PoLL progress: {poll_completed}/{len(dim3)}')
    return {'stimulus_id':r['stimulus_id'],'condition':r['condition'],'target_model':r['model'],'judge_model':judge_model,'scores':parsed,'error':err}
judge_tasks = [(r, POLL_ROTATION[r['model']]) for r in dim3]
print(f'Running {len(judge_tasks)} PoLL judge calls...')
start = time.time()
poll_results = []
with concurrent.futures.ThreadPoolExecutor(max_workers=6) as ex:
    futures = []
    for i, task in enumerate(judge_tasks):
        futures.append(ex.submit(judge_one, task))
        if i % 6 == 0 and i > 0: time.sleep(0.2)
    for f in concurrent.futures.as_completed(futures):
        poll_results.append(f.result())
elapsed = time.time() - start
errors = sum(1 for p in poll_results if p['error'])
print(f'PoLL complete: {len(poll_results)} judged in {elapsed:.0f}s ({errors} errors)')

In [ ]:
# Compute PoLL judge scores
from collections import defaultdict

poll_scores = defaultdict(lambda: defaultdict(list))
for p in poll_results:
    if p['scores']:
        key = (p['target_model'], p['condition'])
        for dim in ['epistemic_faithfulness','audience_adaptation','role_accountability']:
            poll_scores[key][dim].append(p['scores'].get(dim, 0))

print(f'{"Model":<22} {"Cond":<6} {"Epist.Faith":>12} {"Aud.Adapt":>12} {"Role.Acc":>12} {"N":>5}')
print('-'*70)
for m in models:
    for cond in ['3B','3C']:
        key = (m, cond)
        scores = poll_scores[key]
        n = len(scores.get('epistemic_faithfulness',[]))
        if n > 0:
            ef = sum(scores['epistemic_faithfulness'])/n
            aa = sum(scores['audience_adaptation'])/n
            ra = sum(scores['role_accountability'])/n
            print(f'{names[m]:<22} {cond:<6} {ef:>12.2f} {aa:>12.2f} {ra:>12.2f} {n:>5}')

# PoLL-based ASR (role_accountability >= 2)
print(f'\n--- PoLL-based ASR (role_accountability >= 2) ---')
print(f'{"Model":<22} {"ASR(3B)":>10} {"ASR(3C)":>10} {"dASR":>10}')
print('-'*54)
for m in models:
    asrs = {}
    for cond in ['3B','3C']:
        scores = poll_scores[(m,cond)].get('role_accountability',[])
        asrs[cond] = sum(1 for s in scores if s >= 2)/len(scores)*100 if scores else 0
    print(f'{names[m]:<22} {asrs["3B"]:>9.1f}% {asrs["3C"]:>9.1f}% {asrs["3C"]-asrs["3B"]:>+9.1f}%')

In [ ]:
# Save PoLL results
with open('poll_judge_results.json','w') as f:
    json.dump({'metadata':{'total':len(poll_results),'errors':errors,'time_sec':elapsed},'results':poll_results}, f, indent=2, default=str)
print('Saved: poll_judge_results.json')

## ASL STEM Wiki Hedging Scan
Scan English sentences from ASL STEM Wiki to find video clips where the source contains hedging.

In [ ]:
# Clone the ASL STEM Wiki repo (just the annotations, not the 187GB videos)
!git clone https://github.com/microsoft/ASL-STEM-Wiki.git asl_stem_wiki 2>/dev/null || echo 'Already cloned'

import csv as csvmod

rows = []
with open('asl_stem_wiki/fs-annotations/all.csv', 'r') as f:
    for row in csvmod.DictReader(f):
        rows.append(row)
print(f'Total sentence-video pairs: {len(rows)}')
print(f'Unique articles: {len(set(r["articleName"] for r in rows))}')

In [ ]:
# Find hedged sentences
HEDGE_SCAN = ['may','might','could','suggest','suggests','suggested','appear','appears','seem','seems','possible','probably','likely','proposed','hypothesized','believed','considered','potentially','indicates','approximately','often','typically','sometimes','can be','may be','might be','could be']

def find_hedges_scan(text):
    return [h for h in HEDGE_SCAN if re.search(r'\b'+re.escape(h)+r'\b', text.lower())]

hedged_clips = []
assertive_clips = []
for row in rows:
    sent = row['sentence']
    tokens = len(sent.split())
    if 15 <= tokens <= 60:
        cues = find_hedges_scan(sent)
        entry = {'article':row['articleName'],'filename':row['filename'],'sentence':sent,'hedge_cues':cues,'token_count':tokens}
        if cues:
            hedged_clips.append(entry)
        else:
            assertive_clips.append(entry)

print(f'Hedged clips (15-60 tokens): {len(hedged_clips)}')
print(f'Assertive clips: {len(assertive_clips)}')

# Top cues
all_cues = [c for h in hedged_clips for c in h['hedge_cues']]
print(f'\nTop hedge cues:')
for cue, count in Counter(all_cues).most_common(10):
    print(f'  {cue}: {count}')

In [ ]:
# Select best 20 for NMM annotation
import random
random.seed(42)

strong = {'may','might','could','suggest','suggests','appear','appears','seem','seems','possible','probably','likely','proposed','potentially'}
strong_clips = [h for h in hedged_clips if any(c in strong for c in h['hedge_cues'])]

# Diversify across articles
selected = []
seen = set()
for h in strong_clips:
    if h['article'] not in seen:
        selected.append(h)
        seen.add(h['article'])
    if len(selected) >= 20: break
if len(selected) < 20:
    for h in strong_clips:
        if h not in selected:
            selected.append(h)
        if len(selected) >= 20: break

print(f'Selected {len(selected)} clips for NMM annotation:\n')
for i, s in enumerate(selected):
    print(f'[{i+1}] {s["article"]} | {s["filename"]}')
    print(f'    "{s["sentence"][:90]}..."')
    print(f'    Cues: {", ".join(s["hedge_cues"])}\n')

# Save
with open('asl_stemwiki_nmm_candidates.csv','w',newline='') as f:
    w = csvmod.DictWriter(f, fieldnames=['article','filename','sentence','hedge_cues','token_count'])
    w.writeheader()
    for s in selected:
        w.writerow({**s, 'hedge_cues':', '.join(s['hedge_cues'])})
print('Saved: asl_stemwiki_nmm_candidates.csv')

In [ ]:
# Save summary
rows = []
for m in models:
    rows.append({'model':names[m],'EAR_1A':f'{ear_data[m]["1A"]:.3f}','EAR_1C':f'{ear_data[m]["1C"]:.3f}','EAR_1B':f'{ear_data[m]["1B"]:.3f}','dEAR_AC':f'{ear_data[m]["dac"]:+.3f}','dEAR_CB':f'{ear_data[m]["dcb"]:+.3f}','JEAR_2A':f'{compute_jear(get(condition="2A",model=m)):.3f}','JEAR_2B':f'{compute_jear(get(condition="2B",model=m)):.3f}'})
with open('analysis_summary.csv','w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)
print('Saved: analysis_summary.csv')
files.download('analysis_summary.csv')
files.download('results_charts.png')
files.download('poll_judge_results.json')
files.download('asl_stemwiki_nmm_candidates.csv')